In [1]:
# ── CELL 1: Imports ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
})

df_ml = pd.read_csv('../outputs/dataset_with_predictions.csv')
print(f"Loaded: {len(df_ml)} countries")
print(df_ml['risk_category'].value_counts())

Loaded: 88 countries
risk_category
Low Risk       49
Medium Risk    24
High Risk      15
Name: count, dtype: int64


In [2]:
# ── CELL 2: Record your exact PCA weights (from Notebook 2) ───
# Open outputs/dataset_with_mpri.csv — the weights were printed
# in Notebook 2 Cell 4. Re-run this to confirm them here.

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

df_all = pd.read_csv('../outputs/master_dataset.csv')
df2024 = df_all[df_all['year'] == 2024].copy()

df_ml_check = df2024.dropna(
    subset=['wat_basal_t','san_basal_t','hyg_bas_t']
).copy()

features = ['wat_basal_t', 'san_basal_t', 'hyg_bas_t']
scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(df_ml_check[features])

pca      = PCA(n_components=1, random_state=42)
pca.fit(X_scaled)
loadings = np.abs(pca.components_[0])
weights  = loadings / loadings.sum()

print("=" * 50)
print("CONFIRMED PCA-Derived MPRI Weights")
print("Write these into your paper methodology:")
print("=" * 50)
print(f"  Water access weight  : {weights[0]:.4f}  ({weights[0]*100:.1f}%)")
print(f"  Sanitation weight    : {weights[1]:.4f}  ({weights[1]*100:.1f}%)")
print(f"  Hygiene weight       : {weights[2]:.4f}  ({weights[2]*100:.1f}%)")
print(f"  PC1 variance explained: {pca.explained_variance_ratio_[0]*100:.1f}%")
print()
print("MPRI formula for your paper:")
print(f"  MPRI = {weights[0]:.3f}×(100−Water) + {weights[1]:.3f}×(100−Sanitation) + {weights[2]:.3f}×(100−Hygiene)")

CONFIRMED PCA-Derived MPRI Weights
Write these into your paper methodology:
  Water access weight  : 0.2692  (26.9%)
  Sanitation weight    : 0.3630  (36.3%)
  Hygiene weight       : 0.3678  (36.8%)
  PC1 variance explained: 90.2%

MPRI formula for your paper:
  MPRI = 0.269×(100−Water) + 0.363×(100−Sanitation) + 0.368×(100−Hygiene)


In [3]:
# ── CELL 3: Table I — Descriptive Statistics ──────────────────
# This goes directly into your paper as Table I

df_all = pd.read_csv('../outputs/master_dataset.csv')
df2024 = df_all[df_all['year'] == 2024].copy()

stats_cols = {
    'wat_basal_t': 'Basic Water Access (%)',
    'san_basal_t': 'Basic Sanitation (%)',
    'hyg_bas_t':   'Basic Hygiene (%)',
    'wat_basal_r': 'Water Access — Rural (%)',
    'wat_basal_u': 'Water Access — Urban (%)',
    'san_basal_r': 'Sanitation — Rural (%)',
    'san_basal_u': 'Sanitation — Urban (%)',
    'prop_u':      'Urbanisation (%)',
}

rows = []
for col, label in stats_cols.items():
    data = df2024[col].dropna()
    rows.append({
        'Indicator':  label,
        'N':          int(len(data)),
        'Mean':       round(data.mean(), 2),
        'Std Dev':    round(data.std(), 2),
        'Min':        round(data.min(), 2),
        'Median':     round(data.median(), 2),
        'Max':        round(data.max(), 2),
    })

table1 = pd.DataFrame(rows)
print("=" * 85)
print("TABLE I — Descriptive Statistics of WASH Indicators (Global, 2024, n=220 countries)")
print("=" * 85)
print(table1.to_string(index=False))
print()
print("Source: JMP WHO/UNICEF Joint Monitoring Programme, 2025")

# Save as CSV for your paper
table1.to_csv('../outputs/table1_descriptive_statistics.csv', index=False)
print("\nSaved: outputs/table1_descriptive_statistics.csv")

TABLE I — Descriptive Statistics of WASH Indicators (Global, 2024, n=220 countries)
               Indicator   N  Mean  Std Dev   Min  Median   Max
  Basic Water Access (%) 208 91.35    13.52 35.68   97.85 100.0
    Basic Sanitation (%) 207 81.91    25.37 10.36   95.35 100.0
       Basic Hygiene (%)  88 61.41    31.75  3.49   70.40 100.0
Water Access — Rural (%) 165 84.74    18.69 13.82   93.69 100.0
Water Access — Urban (%) 176 95.05     7.93 48.06   98.29 100.0
  Sanitation — Rural (%) 164 73.44    30.50  4.51   87.29 100.0
  Sanitation — Urban (%) 175 83.08    22.42 21.80   95.75 100.0
        Urbanisation (%) 207 63.08    24.53  0.00   65.38 100.0

Source: JMP WHO/UNICEF Joint Monitoring Programme, 2025

Saved: outputs/table1_descriptive_statistics.csv


In [5]:
# ── CELL 4: Figure 20 — World Risk Map ────────────────────────
# Uses plotly — no geopandas needed, just iso3 country codes
# Install first: pip install plotly kaleido

try:
    import plotly.express as px
    import plotly.io as pio
    PLOTLY_AVAILABLE = True
    print("Plotly available — generating interactive world map")
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not installed. Run: pip install plotly kaleido")
    print("Then re-run this cell.")

if PLOTLY_AVAILABLE:
    # Prepare data
    map_data = df_ml[['name','iso3','risk_category','MPRI',
                       'wat_basal_t','san_basal_t','hyg_bas_t']].copy()

    # Assign numeric risk for color ordering
    risk_order = {'Low Risk': 0, 'Medium Risk': 1, 'High Risk': 2}
    map_data['risk_num'] = map_data['risk_category'].map(risk_order)

    color_map = {
        'Low Risk':    '#2E7D32',
        'Medium Risk': '#F57F17',
        'High Risk':   '#C62828'
    }

    fig = px.choropleth(
        map_data,
        locations='iso3',
        color='risk_category',
        color_discrete_map=color_map,
        hover_name='name',
        hover_data={
            'iso3': False,
            'risk_category': True,
            'MPRI': ':.2f',
            'wat_basal_t': ':.1f',
            'san_basal_t': ':.1f',
            'hyg_bas_t': ':.1f',
        },
        category_orders={'risk_category': ['Low Risk','Medium Risk','High Risk']},
        title='Global Marine Pollution Risk Assessment — 2024<br>'
              '<sup>PCA-Weighted MPRI from WASH Indicators | JMP WHO/UNICEF 2025 | n=88 countries</sup>',
    )

    fig.update_layout(
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor='#BDBDBD',
            showland=True,
            landcolor='#F5F5F5',
            showocean=True,
            oceancolor='#E3F2FD',
            showlakes=False,
            projection_type='natural earth'
        ),
        legend=dict(
            title='Risk Category',
            orientation='h',
            y=-0.05,
            x=0.5,
            xanchor='center'
        ),
        margin=dict(l=0, r=0, t=60, b=0),
        height=500,
        font=dict(family='Arial', size=12)
    )

    # Save interactive HTML
    fig.write_html('../figures/fig20_world_risk_map.html')
    print("Saved: figures/fig20_world_risk_map.html  (interactive)")

    # Save static PNG for paper
    try:
        fig.write_image('../figures/fig20_world_risk_map.png',
                        width=1400, height=700, scale=2)
        print("Saved: figures/fig20_world_risk_map.png  (for paper)")
    except Exception as e:
        print(f"PNG export needs kaleido: pip install kaleido")
        print(f"Error: {e}")

    fig.show()

Plotly available — generating interactive world map
Saved: figures/fig20_world_risk_map.html  (interactive)
Saved: figures/fig20_world_risk_map.png  (for paper)


In [6]:
# ── CELL 5: Figure 21 — Sample Policy Recommendations Table ───
# A visual summary table for your paper's results section

df_recs = pd.read_csv('../outputs/policy_recommendations.csv')

# Select 6 representative countries (2 per risk category)
high_risk_sample   = df_recs[df_recs['risk_category']=='High Risk'].nlargest(2, 'mpri_score')
medium_risk_sample = df_recs[df_recs['risk_category']=='Medium Risk'].iloc[[0,12]]
low_risk_sample    = df_recs[df_recs['risk_category']=='Low Risk'][
    df_recs['country'].isin(['Sri Lanka','Brazil'])
]
if len(low_risk_sample) < 2:
    low_risk_sample = df_recs[df_recs['risk_category']=='Low Risk'].head(2)

sample = pd.concat([high_risk_sample, medium_risk_sample, low_risk_sample])

print("=" * 90)
print("TABLE II — Sample SHAP-Driven Policy Recommendations by Risk Category")
print("=" * 90)
for _, row in sample.iterrows():
    print(f"\nCountry       : {row['country']}")
    print(f"Risk Category : {row['risk_category']}  |  MPRI: {row['mpri_score']}  |  Urgency: {row['urgency']}")
    print(f"Key Driver    : {row['top_shap_driver']}")
    print(f"Primary Action: {row['primary_action'][:110]}...")
    print("-" * 90)

# Save for paper
sample[['country','risk_category','urgency','mpri_score',
        'top_shap_driver','primary_action']].to_csv(
    '../outputs/table2_policy_sample.csv', index=False
)
print("\nSaved: outputs/table2_policy_sample.csv")

TABLE II — Sample SHAP-Driven Policy Recommendations by Risk Category

Country       : Ethiopia
Risk Category : High Risk  |  MPRI: 100.0  |  Urgency: URGENT
Key Driver    : Sanitation (rural)
Primary Action: Immediate construction of rural sanitation facilities and wastewater treatment infrastructure to prevent untre...
------------------------------------------------------------------------------------------

Country       : Central African Republic
Risk Category : High Risk  |  MPRI: 97.11  |  Urgency: URGENT
Key Driver    : Sanitation (rural)
Primary Action: Immediate construction of rural sanitation facilities and wastewater treatment infrastructure to prevent untre...
------------------------------------------------------------------------------------------

Country       : Afghanistan
Risk Category : Medium Risk  |  MPRI: 49.73  |  Urgency: MODERATE
Key Driver    : Water ARC (annual change)
Primary Action: Accelerate progress on water arc (annual change) to prevent regression to

In [7]:
# ── CELL 6: Final completeness check ─────────────────────────
import os

print("=" * 65)
print("FINAL PROJECT COMPLETENESS CHECK")
print("=" * 65)

figures_expected = [f'fig{i}' for i in range(1, 22)]
figures_found    = [f for f in os.listdir('../figures') if f.endswith('.png') or f.endswith('.html')]

print(f"\nFigures in figures/ folder: {len(figures_found)}")
for f in sorted(figures_found):
    print(f"  {f}")

outputs_expected = [
    'master_dataset.csv',
    'dataset_2024.csv',
    'srilanka_data.csv',
    'dataset_with_mpri.csv',
    'dataset_with_predictions.csv',
    'final_dataset_complete.csv',
    'policy_recommendations.csv',
    'cv_results_table.csv',
    'table1_descriptive_statistics.csv',
    'table2_policy_sample.csv',
    'rf_model.pkl',
    'xgb_model.pkl',
]

print(f"\nOutputs check:")
for f in outputs_expected:
    exists = os.path.exists(f'../outputs/{f}')
    status = "OK" if exists else "MISSING"
    print(f"  {status:7s}  {f}")

print()
print("=" * 65)
print("COMPLETE. You are ready to write the IEEE paper.")
print("=" * 65)
print(f"""
Key numbers to use in your paper:
  Dataset    : JMP WHO/UNICEF 2025, 233 countries, 2000-2024
  ML dataset : 88 countries (complete WASH data, 2024 snapshot)
  MPRI       : PCA-weighted — Water {weights[0]:.3f}, San {weights[1]:.3f}, Hyg {weights[2]:.3f}
  PC1 var    : {pca.explained_variance_ratio_[0]*100:.1f}%
  Clustering : K-Means k=3, silhouette = 0.5997
               Low=49, Medium=24, High=15 countries
  RF model   : Accuracy=0.9549, F1=0.9257 (5-fold CV, macro)
  XGBoost    : Accuracy=0.9196, F1=0.8758 (5-fold CV, macro)
  Top SHAP   : Sanitation rural > Sanitation national > Hygiene national
  Sri Lanka  : Low Risk, MPRI=11.16, rank 61/88
               Sanitation rural (-0.083) strongest protective factor
               Only urbanisation (+0.002) slightly raises risk
""")

FINAL PROJECT COMPLETENESS CHECK

Figures in figures/ folder: 21
  fig10_regional_mpri.png
  fig11_confusion_matrices.png
  fig12_feature_importance.png
  fig13_cv_comparison.png
  fig14_feature_importance_comparison.png
  fig15_shap_summary_bar.png
  fig16_shap_beeswarm.png
  fig17_srilanka_waterfall.png
  fig18_shap_dependence.png
  fig19_policy_dashboard.png
  fig1_wash_distributions.png
  fig20_world_risk_map.html
  fig20_world_risk_map.png
  fig2_srilanka_trend.png
  fig3_income_comparison.png
  fig4_rural_urban_gap.png
  fig5_correlation_heatmap.png
  fig6_mpri_distribution.png
  fig7_optimal_k.png
  fig8_cluster_results.png
  fig9_srilanka_mpri_trend.png

Outputs check:
  OK       master_dataset.csv
  OK       dataset_2024.csv
  OK       srilanka_data.csv
  OK       dataset_with_mpri.csv
  OK       dataset_with_predictions.csv
  OK       final_dataset_complete.csv
  OK       policy_recommendations.csv
  OK       cv_results_table.csv
  OK       table1_descriptive_statistics.csv
 